# MNIST Dataset

Dataset: (https://www.tensorflow.org/datasets/catalog/mnist).

Each sample has a 28x28 grayscale image of a single handwritten digit (0-9), and the corresponding expected output, again 0-9.


In [ ]:
from datetime import datetime
import os
import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow.keras.layers import Input, SimpleRNN, Dense, TimeDistributed
from tensorflow.keras.models import Model

## Data Processing

The dataset is already loaded and split into a training and a testing set.

You can directly use it for a tensorflow / keras model.


In [ ]:
# Enable GPU memory growth if GPUs are available
gpus = tf.config.experimental.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

# Load MNIST dataset
ds_train, ds_val, ds_test = tfds.load(
    "mnist",
    split=["train[:80%]", "train[80%:]", "test"],
    as_supervised=True,
    shuffle_files=True
)

# Normalize pixel values from [0, 255] to [0.0, 1.0]
normalize = lambda x, y: (tf.cast(x, tf.float32) / 255.0, tf.cast(y, tf.int32))
ds_train = ds_train.map(normalize)
ds_val   = ds_val.map(normalize)
ds_test  = ds_test.map(normalize)

In [ ]:
def prepare_sequences(x, y):
    """
    Task: 
    1. Reshape the image dimensions: From (28, 28, 1) to a time-series of (28, 28).
    2. Replicate the single label 'y' so that a label exists for each of the 
       28 time steps (Many-to-Many approach).
    """
    # -----------------------------------------------------------------
    # TODO: Adjust image dimensions and replicate the label
    # x_seq = ...
    # y_seq = ...
    # -----------------------------------------------------------------

    return x_seq, y_seq

BATCH_SIZE = 64
train_sequence_ds = ds_train.map(prepare_sequences).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_sequence_ds   = ds_val.map(prepare_sequences).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_sequence_ds  = ds_test.map(prepare_sequences).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [ ]:
def build_anytime_rnn():
    """
    Task: Build an RNN that performs a classification after each individual 
    time step (each scanned row). Use the TimeDistributed class to achieve this.
    """
    # Input interface for (28 time steps, 28 pixels per row)
    inputs = Input(shape=(28, 28))

    # -----------------------------------------------------------------
    # TODO: 
    # 1. Add a recurrent layer (e.g., SimpleRNN or LSTM).
    #    CRITICAL: Consider which parameter is required for Many-to-Many! (Paramater: return_sequences)
    # 2. Add one or more MLP Dense layers wrapped inside TimeDistributed.
    # 3. The output must be a Softmax layer with 10 outputs (also wrapped in TimeDistributed).
    # -----------------------------------------------------------------

    model = Model(inputs=inputs, outputs=outputs, name="Anytime_MNIST_RNN")
    return model

model = build_anytime_rnn()
model.summary()

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("\n--- Training Started ---")
model.fit(train_sequence_ds, epochs=3, validation_data=val_sequence_ds)

In [ ]:
def evaluate_with_threshold(model, dataset, confidence_threshold=0.8):
    """
    Task: Evaluate the model manually batch by batch.
    Stop processing predictions for an image as soon as the confidence score 
    (Softmax probability) meets or exceeds the 'confidence_threshold'.
    
    Return the overall final accuracy and the average time step (scanned row) 
    at which the decision was made.
    """
    total_samples = 0
    correct_predictions = 0
    total_time_steps_taken = 0

    print(f"\nEvaluating with Confidence Threshold: {confidence_threshold}")

    for x_batch, y_batch in dataset:
        predictions = model.predict_on_batch(x_batch)
        true_labels = y_batch[:, 0].numpy() # Since all 28 rows share the same true label

        for i in range(len(predictions)):
            sample_preds = predictions[i] # Predictions for a single image (28, 10)
            true_label = true_labels[i]

            # Default value if the threshold is never reached (falls back to the last row)
            chosen_step = 27 
            predicted_class = np.argmax(sample_preds[-1])

            # -----------------------------------------------------------------
            # TODO:
            # Iterate through the time steps t (0 to 27).
            # Determine the maximum probability (confidence) at step t.
            # If this value is >= confidence_threshold:
            #   -> Save the current time step (t)
            #   -> Determine the predicted class at this specific step
            #   -> Break the loop for this image (break)
            # -----------------------------------------------------------------
            for t in range(28):
                # Insert code here
                pass

            # -----------------------------------------------------------------
            total_samples += 1
            total_time_steps_taken += (chosen_step + 1) # +1 for 1-based indexing (Row 1 to 28)
            if predicted_class == true_label:
                correct_predictions += 1

    avg_step = total_time_steps_taken / total_samples
    accuracy = correct_predictions / total_samples

    print(f"-> Accuracy: {accuracy:.4f}")
    print(f"-> Average row steps until decision: {avg_step:.2f} / 28")

    return accuracy, avg_step

thresholds_to_test = [0.5, 0.7, 0.9, 0.98]
for thresh in thresholds_to_test:
    evaluate_with_threshold(model, test_sequence_ds, confidence_threshold=thresh)